# omnia-20x — does it actually make training faster?

This notebook measures **end-to-end training time**, not just data loading, and
reports *why* you got the number you got.

The honest framing: omnia-20x removes data-loading cost. How much that speeds up
**training** depends entirely on what fraction of your epoch was data loading in
the first place. If your GPU is already the bottleneck, removing all I/O still
cannot help much — that is Amdahl's law, not a limitation of the format.

This notebook measures that fraction and tells you your ceiling before it tells
you your speedup.

**Runtime → Change runtime type → GPU** before running.


## 1. Setup


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — set Runtime > Change runtime type > GPU'


In [ ]:
%%bash
pip install -q numpy torch torchvision pillow zstandard tifffile openslide-python 2>&1 | tail -1
apt-get -qq install -y openslide-tools > /dev/null 2>&1
echo 'deps installed'


## 2. Install omnia-20x

Upload the `omnia-20x` folder to `/content/` (Files pane → upload folder), or
zip it and upload the zip. The cell handles either.


In [ ]:
import os, glob, subprocess, sys

root = None
for cand in ('/content/omnia-20x', '/content/omnia_20x'):
    if os.path.isdir(cand): root = cand; break
if root is None:
    z = glob.glob('/content/*omnia*20x*.zip') + glob.glob('/content/*omnia*.zip')
    if z:
        subprocess.run(['unzip','-q','-o',z[0],'-d','/content'], check=True)
        for cand in glob.glob('/content/**/pyproject.toml', recursive=True):
            if 'omnia_sdk' in os.listdir(os.path.dirname(cand)):
                root = os.path.dirname(cand); break

assert root, 'Upload the omnia-20x folder (or a zip of it) to /content first.'
print('found:', root)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',root], check=True)

import omnia_sdk
print('omnia_sdk', omnia_sdk.__version__, 'installed')


## 3. Get a slide

Downloads a public 178 MB Aperio test slide, so this runs with no upload and no
Google Drive. Point `SVS_PATH` at your own slide instead if you prefer.


In [ ]:
SVS_PATH = '/content/CMU-1.svs'
if not os.path.exists(SVS_PATH):
    !wget -q --show-progress -O {SVS_PATH} https://openslide.cs.cmu.edu/download/openslide-testdata/Aperio/CMU-1.svs

import openslide
s = openslide.OpenSlide(SVS_PATH)
print(f'{os.path.getsize(SVS_PATH)/1e6:.0f} MB · {s.dimensions[0]}x{s.dimensions[1]} px · {s.level_count} levels')
s.close()


## 4. Convert to .omnia

`--min-level 1` drops the 40x level. That keeps the preload inside Colab's RAM —
full resolution would need far more than the free tier provides.


In [ ]:
import time
OMNIA_PATH = '/content/slide.train.omnia'
t0 = time.time()
r = subprocess.run([sys.executable,'-m','omnia_sdk.cli','svs-convert',
                    SVS_PATH, OMNIA_PATH,'--batch-size','16','--zstd-level','5','--min-level','1'],
                   capture_output=True, text=True)
print(r.stdout[-600:])
assert r.returncode == 0, r.stderr[-800:]
print(f'converted in {time.time()-t0:.1f}s -> {os.path.getsize(OMNIA_PATH)/1e6:.1f} MB')


## 5. The benchmark

Identical model, batch size, shuffling and epoch count on both sides. The only
difference is where tiles come from. Epoch 1 is discarded on both sides so neither
is charged for cache warm-up.

Crucially this separates **data time** from **compute time** inside each epoch,
because that split is what determines your speedup.


In [ ]:
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS, BATCH, TILE = 4, 64, 256
print('device:', torch.cuda.get_device_name() if device.type=='cuda' else 'CPU (expect a small speedup)')

class SvsTiles(Dataset):
    """Status quo: openslide decodes JPEG-2000 during training, every epoch."""
    def __init__(self, path, level=1, tile=TILE):
        self.path, self.level, self.tile, self.s = path, level, tile, None
        sl = openslide.OpenSlide(path)
        w, h = sl.level_dimensions[level]; self.ds = sl.level_downsamples[level]; sl.close()
        self.tx, self.ty = (w+tile-1)//tile, (h+tile-1)//tile
    def __len__(self): return self.tx*self.ty
    def __getitem__(self, i):
        if self.s is None: self.s = openslide.OpenSlide(self.path)   # per-worker handle
        x = int((i % self.tx)*self.tile*self.ds); y = int((i//self.tx)*self.tile*self.ds)
        a = np.asarray(self.s.read_region((x,y), self.level, (self.tile,self.tile)).convert('RGB'))
        return torch.from_numpy(a.copy()).permute(2,0,1).float()/255.0, 0

def run(ds, label):
    m = resnet18(weights=None); m.fc = nn.Linear(512,2); m = m.to(device).train()
    opt = torch.optim.SGD(m.parameters(), lr=0.01); crit = nn.CrossEntropyLoss()
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=(device.type=='cuda'))
    ep, dt = [], []
    for e in range(EPOCHS):
        if device.type=='cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); data_t = 0.0; it = iter(dl)
        while True:
            td = time.perf_counter()
            try: x,y = next(it)
            except StopIteration: break
            data_t += time.perf_counter()-td
            x = x.to(device, non_blocking=True); y = torch.as_tensor(y).to(device)
            opt.zero_grad(); crit(m(x), y).backward(); opt.step()
        if device.type=='cuda': torch.cuda.synchronize()
        tot = time.perf_counter()-t0
        ep.append(tot); dt.append(data_t)
        print(f'  {label} epoch {e+1}: {tot:6.2f}s   data {data_t:5.2f}s ({data_t/tot*100:4.1f}%)')
    return float(np.mean(ep[1:])), float(np.mean(dt[1:]))   # discard epoch 1


In [ ]:
from omnia_sdk.dataset import OmniaDataset

t0 = time.time(); ds_om = OmniaDataset(OMNIA_PATH, cache_mode='ram'); preload = time.time()-t0
print(f'preload: {preload:.1f}s for {len(ds_om)} tiles\n')

# Same tile count on both sides, or the comparison is meaningless.
N = min(len(ds_om), len(SvsTiles(SVS_PATH)))
ds_svs = Subset(SvsTiles(SVS_PATH), range(N))
print(f'benchmarking {N} tiles/epoch, {EPOCHS} epochs\n')

svs_ep, svs_dt = run(ds_svs, '.svs  ')
print()
om_ep,  om_dt  = run(Subset(ds_om, range(N)), '.omnia')


## 6. Results — and why


In [ ]:
data_share = svs_dt/svs_ep
ceiling    = 1/(1-data_share) if data_share < 1 else float('inf')
e2e        = svs_ep/om_ep
dataspeed  = svs_dt/om_dt if om_dt > 0 else float('inf')
regime     = 'DATA-BOUND' if e2e >= 10 else ('MIXED' if e2e >= 2 else 'MODEL-BOUND')

print('='*60)
print(f'  GPU                    : {torch.cuda.get_device_name() if device.type=="cuda" else "CPU"}')
print(f'  tiles/epoch            : {N}')
print('-'*60)
print(f'  .svs   epoch           : {svs_ep:6.2f}s   (data {svs_dt:5.2f}s = {data_share*100:.1f}%)')
print(f'  .omnia epoch           : {om_ep:6.2f}s   (data {om_dt:5.2f}s = {om_dt/om_ep*100:.1f}%)')
print('-'*60)
print(f'  DATA LOADING           : {dataspeed:6.1f}x faster')
print(f'  END-TO-END TRAINING    : {e2e:6.2f}x faster        <-- the number that matters')
print(f'  theoretical ceiling    : {ceiling:6.2f}x  (Amdahl, from {data_share*100:.1f}% data share)')
print(f'  regime                 : {regime}')
print('-'*60)
print(f'  preload (one-off)      : {preload:.1f}s')
print(f'  breaks even after      : {preload/(svs_ep-om_ep):.1f} epochs' if svs_ep>om_ep else '  preload never repaid on this hardware')
print('='*60)
# Compute time is the same model on the same GPU, so it must match across both
# runs. If it does not, the machine was doing something else and the comparison
# is noise -- say so rather than reporting a number that will not reproduce.
svs_compute, om_compute = svs_ep - svs_dt, om_ep - om_dt
skew = abs(svs_compute - om_compute) / max(svs_compute, om_compute)
print()
if skew > 0.20:
    print(f'!! UNRELIABLE: compute time differed by {skew*100:.0f}% between the two runs')
    print(f'   ({svs_compute:.2f}s vs {om_compute:.2f}s) -- it should be identical.')
    print('   Too few tiles/epochs, or the machine is busy. Raise EPOCHS or use a')
    print('   bigger slide, and re-run before trusting the end-to-end number.')
    print()

if data_share < 0.5:
    print('Data loading was under half your epoch, so the ceiling is low no matter')
    print('how fast the container is. The GPU is your bottleneck here, not I/O.')
else:
    print('Data loading dominated the epoch, which is exactly the case omnia-20x')
    print('is built for — most of that cost is now gone.')


---

### Reading this honestly

**Data-loading speedup** is a property of the format and will be large on any
machine. **End-to-end speedup** is capped by how much of your epoch was I/O:

| data share of epoch | best possible end-to-end |
|---|---|
| 90% | 10x |
| 75% | 4x |
| 50% | 2x |
| 37% | 1.6x |

A small end-to-end number does not mean the container underperformed — it means
there was little I/O left to remove. Check the *ceiling* line before judging the
*speedup* line.

Measured on Apple M5 (MPS) for reference: data loading 15.2x, end-to-end 1.31x,
data share 37% — a model-bound machine. A CUDA GPU should shift this well toward
data-bound.
